# Day 029 — Exercise 5: build_notification

**What you'll build:** `build_notification(event_type, data, model='llama3.2') -> dict` — the pipeline capstone. Takes an event type and data dict, generates an AI summary, and returns a dict with Slack and Discord payloads ready to send.

**Why it matters:** build_notification is the single entry point for turning any automation event into a multi-platform notification. One call produces payloads for both Slack and Discord — the NotificationBot just adds the HTTP send step.

In [ ]:
import ollama

## Provided: Helper Functions

In [ ]:
def format_slack_message(
    title: str,
    body: str,
    color: str = "#36a64f",
) -> dict:
    from datetime import datetime
    return {
        "attachments": [
            {
                "fallback": title,
                "color":    color,
                "title":    title,
                "text":     body,
                "footer":   "NotificationBot",
                "ts":       int(datetime.now().timestamp()),
            }
        ]
    }


def format_discord_embed(
    title: str,
    description: str,
    color: int = 0x00b0f4,
) -> dict:
    return {
        "title":       title,
        "description": description,
        "color":       color,
    }


def truncate_for_chat(text: str, max_chars: int = 2000) -> str:
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."


def ai_summarize_for_chat(
    content: str,
    platform: str = "slack",
    model: str = "llama3.2",
) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are a notification writer for {platform}. "
                    "Write a concise notification summary: under 300 characters, "
                    "no headers, no bullet points. Lead with the most important fact."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Summarise this for a {platform} notification:\n\n"
                    f"{content[:3000]}"
                ),
            },
        ],
    )
    return response["message"]["content"]

## Your Implementation

In [ ]:
def build_notification(
    event_type: str,
    data: dict,
    model: str = 'llama3.2',
) -> dict:
    """
    Build Slack and Discord notification payloads for an event.

    Args:
        event_type: Dot-notation event identifier (e.g. 'report.generated').
        data:       Dict of event details {key: value}.
        model:      Ollama model name.

    Returns:
        Dict with keys: event_type, title, summary,
                        slack_payload, discord_payload.
    """
    # TODO: content = f'Event: {event_type}\n\nData:\n'
    #        + '\n'.join(f'  {k}: {v}' for k, v in data.items())
    # TODO: summary = ai_summarize_for_chat(content, platform='slack', model=model)
    # TODO: title = f'[{event_type.upper()}] Notification'
    # TODO: slack_payload = format_slack_message(title, summary)
    # TODO: discord_payload = {'embeds': [format_discord_embed(title, truncate_for_chat(summary, 4096))]}
    # TODO: return {event_type, title, summary, slack_payload, discord_payload}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    EVENT = 'report.generated'
    DATA  = {'rows': 500, 'output': '/tmp/report.xlsx', 'duration_s': 12}

    # Check 1: defined
    try:
        assert 'build_notification' in globals()
        passed += 1; print('\u2705 Check 1: build_notification defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    result = None

    # Check 2: returns a dict with all required keys
    try:
        result = build_notification(EVENT, DATA)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result)}'
        for k in ('event_type', 'title', 'summary', 'slack_payload', 'discord_payload'):
            assert k in result, f"result missing '{k}': {list(result)}"
        passed += 1; print('\u2705 Check 2: all required keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: event_type and title correct
    try:
        assert result is not None
        assert result['event_type'] == EVENT, \
            f"event_type wrong: {result['event_type']!r}"
        assert result['title'] == '[REPORT.GENERATED] Notification', \
            f"title wrong: {result['title']!r}"
        passed += 1; print(f"\u2705 Check 3: event_type and title correct")
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: slack_payload has 'attachments' structure
    try:
        assert result is not None
        sp = result['slack_payload']
        assert 'attachments' in sp, \
            f"slack_payload missing 'attachments': {list(sp)}"
        att = sp['attachments'][0]
        assert att.get('title') == result['title'], \
            f"slack title mismatch: {att.get('title')!r}"
        passed += 1; print('\u2705 Check 4: slack_payload has correct attachments')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: discord_payload has 'embeds' structure
    try:
        assert result is not None
        dp = result['discord_payload']
        assert 'embeds' in dp, \
            f"discord_payload missing 'embeds': {list(dp)}"
        embed = dp['embeds'][0]
        assert embed.get('title') == result['title'], \
            f"discord embed title mismatch: {embed.get('title')!r}"
        passed += 1; print('\u2705 Check 5: discord_payload has correct embeds')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def build_notification(
    event_type: str,
    data: dict,
    model: str = "llama3.2",
) -> dict:
    content = (
        f"Event: {event_type}\n\nData:\n"
        + "\n".join(f"  {k}: {v}" for k, v in data.items())
    )
    summary = ai_summarize_for_chat(content, platform="slack", model=model)
    title   = f"[{event_type.upper()}] Notification"
    return {
        "event_type":      event_type,
        "title":           title,
        "summary":         summary,
        "slack_payload":   format_slack_message(title, summary),
        "discord_payload": {
            "embeds": [format_discord_embed(title, truncate_for_chat(summary, 4096))]
        },
    }
```

</details>